# D.O.N.G — Huấn luyện & Đánh giá MLP trên Google Colab
**Học phần:** IT3190 · Nhập môn Học máy & Khai phá dữ liệu  
**Phụ trách MLP:** Đinh Ngọc Khánh (20235752)  

Notebook chạy trực tiếp trên **Google Colab** với file `archive.zip` (hoặc `cybersecurity_attacks.csv`) theo đúng kế hoạch đồ án D.O.N.G:
1. Nạp dữ liệu & kiểm tra nhanh (EDA: phân bố lớp, thiếu, trùng).
2. Tiền xử lý + Feature Engineering + **chặn Data Leakage** (bỏ cột leaky & cột định danh).
3. Chia dữ liệu **60 / 20 / 20 phân tầng** (`random_state=42`); **không mở tập Test** ở giai đoạn báo cáo tiến độ 25/11.
4. Huấn luyện **Baseline** vs **MLP v1** (`Pipeline` = `StandardScaler` + `OneHotEncoder` + `MLPClassifier` dùng ReLU, Adam, Early Stopping).
5. Vẽ **Learning Curve**, **Confusion Matrix**, thử lưới tham số + **3 seeds** và xuất `results_mlp_progress.csv` + `mlp_v1.joblib`.


In [ ]:
# 1. Import thư viện
import os
import time
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("Import thành công!")


---
## Bước 1: Nạp dữ liệu (`archive.zip` hoặc `cybersecurity_attacks.csv`)
Chạy cell dưới đây trên Google Colab → bấm **Choose Files** và chọn file `archive.zip` (hoặc `cybersecurity_attacks.csv`) từ máy tính của bạn.


In [ ]:
CSV_PATH = "cybersecurity_attacks.csv"

# Tự động bật hộp thoại upload nếu đang chạy trên Google Colab và chưa có file
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("Hãy chọn file archive.zip hoặc cybersecurity_attacks.csv...")
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                with zipfile.ZipFile(fname, "r") as z:
                    z.extractall(".")
                print(f"Đã giải nén {fname}!")
            elif fname.endswith(".csv"):
                CSV_PATH = fname
    except ImportError:
        raise FileNotFoundError(
            "Không tìm thấy cybersecurity_attacks.csv. Hãy đặt file CSV cùng thư mục hoặc chạy trên Google Colab."
        )

df = pd.read_csv(CSV_PATH)
print("Kích thước dữ liệu:", df.shape)
df.head(3)


---
## Bước 2: EDA nhanh (Phân bố lớp, giá trị thiếu, bản ghi trùng)


In [ ]:
print("Số dòng trùng hoàn toàn:", df.duplicated().sum())
print("\nPhân bố 3 lớp mục tiêu (Attack Type):")
print(df["Attack Type"].value_counts())

print("\nCác cột có giá trị thiếu:")
nulls = df.isna().sum()
print(nulls[nulls > 0])


---
## Bước 3: Tiền xử lý, Feature Engineering & Chống Data Leakage
Thực hiện đúng **Mục 3** trong kế hoạch đồ án D.O.N.G:
- **Loại cột leaky** (chỉ có sau khi tấn công đã được phát hiện): `Attack Signature`, `Action Taken`, `Severity Level`, `IDS/IPS Alerts`.
- **Loại cột định danh / văn bản lorem ipsum**: `Source IP Address`, `Destination IP Address`, `User Information`, `Payload Data`, `Geo-location Data`, `Proxy Information` (chỉ giữ cờ nhị phân `has_proxy`).
- **Tạo đặc trưng mới**: giờ/thứ/tháng từ `Timestamp`; dải cổng (`well_known` / `registered` / `dynamic`); hệ điều hành & trình duyệt từ `Device Information`.


In [ ]:
def port_band(port: int) -> str:
    if port < 1024:
        return "well_known"
    if port < 49152:
        return "registered"
    return "dynamic"


def extract_os(ua: str) -> str:
    s = str(ua).lower()
    if "windows" in s:
        return "Windows"
    if "android" in s:
        return "Android"
    if "iphone" in s or "ipad" in s or "ios" in s:
        return "iOS"
    if "mac os" in s or "macintosh" in s:
        return "macOS"
    if "linux" in s:
        return "Linux"
    return "Other"


def extract_browser(ua: str) -> str:
    s = str(ua).lower()
    if "chrome" in s and "edg" not in s:
        return "Chrome"
    if "firefox" in s:
        return "Firefox"
    if "safari" in s and "chrome" not in s:
        return "Safari"
    if "msie" in s or "trident" in s or "edg" in s:
        return "IE_Edge"
    if "opera" in s or "opr" in s:
        return "Opera"
    return "Other"


def build_features(raw_df: pd.DataFrame):
    data = raw_df.drop_duplicates().copy()

    # 1. Đặc trưng thời gian
    ts = pd.to_datetime(data["Timestamp"], errors="coerce")
    data["hour"] = ts.dt.hour.fillna(0).astype(int)
    data["dayofweek"] = ts.dt.dayofweek.fillna(0).astype(int)
    data["month"] = ts.dt.month.fillna(1).astype(int)

    # 2. Nhóm dải cổng
    data["src_port_band"] = data["Source Port"].apply(port_band)
    data["dst_port_band"] = data["Destination Port"].apply(port_band)

    # 3. Tách OS & Browser từ Device Information
    data["os_family"] = data["Device Information"].apply(extract_os)
    data["browser_family"] = data["Device Information"].apply(extract_browser)

    # 4. Xử lý giá trị thiếu (rỗng -> "None") & cờ Proxy
    data["has_proxy"] = data["Proxy Information"].notna().astype(int)
    for col in ["Malware Indicators", "Alerts/Warnings", "Firewall Logs"]:
        data[col] = data[col].fillna("None")

    num_cols = [
        "Source Port",
        "Destination Port",
        "Packet Length",
        "Anomaly Scores",
        "hour",
        "dayofweek",
        "month",
        "has_proxy",
    ]
    cat_cols = [
        "Protocol",
        "Packet Type",
        "Traffic Type",
        "Malware Indicators",
        "Alerts/Warnings",
        "Network Segment",
        "Firewall Logs",
        "Log Source",
        "src_port_band",
        "dst_port_band",
        "os_family",
        "browser_family",
    ]

    X = data[num_cols + cat_cols].copy()
    le = LabelEncoder()
    y = le.fit_transform(data["Attack Type"])
    class_names = list(le.classes_)
    return X, y, num_cols, cat_cols, le, class_names


X, y, num_cols, cat_cols, le, CLASS_NAMES = build_features(df)
print(f"Số đặc trưng sau khi chặn leakage: {X.shape[1]} ({len(num_cols)} số + {len(cat_cols)} phân loại)")
print("Ánh xạ nhãn:", dict(enumerate(CLASS_NAMES)))
X.head(3)


---
## Bước 4: Chia dữ liệu 60 / 20 / 20 phân tầng (`random_state = 42`)
- **Train (60%):** 24.000 dòng — huấn luyện mô hình.
- **Validation (20%):** 8.000 dòng — chọn tham số & báo cáo tiến độ 25/11.
- **Test (20%):** 8.000 dòng — **KHÓA**, chỉ mở đúng 1 lần vào ngày 6/12!


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test (locked): {X_test.shape}")


---
## Bước 5: Huấn luyện Baseline & MLP v1 (trong `sklearn.Pipeline`)
Mọi bước chuẩn hóa (`StandardScaler`) và mã hóa (`OneHotEncoder`) đều nằm **bên trong `Pipeline`** nên chỉ `fit` trên tập Train — đảm bảo **0% data leakage**.


In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ]
    )


def evaluate_on_val(name, pipe, X_tr, y_tr, X_v, y_v):
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    train_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = pipe.predict(X_v)
    pred_s = time.perf_counter() - t0

    f1 = f1_score(y_v, y_pred, average="macro")
    acc = accuracy_score(y_v, y_pred)
    print(f"[{name:24s}] Val Macro-F1 = {f1:.4f} | Acc = {acc:.4f} | train = {train_s:.2f}s | pred = {pred_s:.4f}s")
    return {
        "model": name,
        "val_macro_f1": round(f1, 4),
        "val_accuracy": round(acc, 4),
        "train_time_s": round(train_s, 3),
        "pred_time_s": round(pred_s, 4),
    }, y_pred


results = []

# 1. Baseline đoán lớp đông nhất
dummy_pipe = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", DummyClassifier(strategy="most_frequent")),
])
res_dummy, _ = evaluate_on_val("Baseline (Majority)", dummy_pipe, X_train, y_train, X_val, y_val)
results.append(res_dummy)

# 2. MLP v1 (ReLU + Adam + Early Stopping)
mlp_v1 = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        batch_size=256,
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=42,
    )),
])
res_mlp1, y_val_pred = evaluate_on_val("MLP v1 (128, 64)", mlp_v1, X_train, y_train, X_val, y_val)
results.append(res_mlp1)

print("\nClassification Report — MLP v1 (Validation):")
print(classification_report(y_val, y_val_pred, target_names=CLASS_NAMES, digits=4))


---
## Bước 6: Vẽ Learning Curve & Confusion Matrix (dùng cho báo cáo tiến độ)


In [ ]:
mlp_clf = mlp_v1.named_steps["clf"]
classes = list(mlp_clf.classes_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. Learning Curve
ax1 = axes[0]
ax1.plot(mlp_clf.loss_curve_, color="#c00005", linewidth=2, label="Train Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss", color="#c00005")
ax1.tick_params(axis="y", labelcolor="#c00005")
ax1.set_title("MLP v1 — Learning Curve (Early Stopping)")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(mlp_clf.validation_scores_, color="#006492", linewidth=2, linestyle="--", label="Val Score")
ax2.set_ylabel("Validation Accuracy (internal)", color="#006492")
ax2.tick_params(axis="y", labelcolor="#006492")

# 2. Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred, labels=list(range(len(CLASS_NAMES))))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title("MLP v1 — Confusion Matrix (Validation)")

plt.tight_layout()
plt.savefig("mlp_v1_learning_and_cm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu hình -> mlp_v1_learning_and_cm.png")


---
## Bước 7: Thử nghiệm các kiến trúc MLP khác nhau & Chạy 3 Random Seeds
Đúng theo **Mục 5** kế hoạch: thử 1–2 tầng ẩn `{64, 128}`, điều chỉnh `alpha` (L2), chạy 3 seeds (`42, 123, 2026`) để báo cáo dạng **trung bình ± độ lệch chuẩn**, và lưu model tốt nhất bằng `joblib`.


In [ ]:
configs = [
    ("MLP (64,) a=1e-4",      (64,),     1e-4, 1e-3),
    ("MLP (128,) a=1e-3",     (128,),    1e-3, 1e-3),
    ("MLP (128, 64) a=1e-3",  (128, 64), 1e-3, 1e-3),
    ("MLP (128, 64) a=1e-2",  (128, 64), 1e-2, 1e-3),
]
seeds = [42, 123, 2026]

summary_rows = []
best_f1 = -1.0
best_pipe = None
best_name = ""

for name, hidden, alpha, lr in configs:
    f1s, accs, t_trains = [], [], []
    for sd in seeds:
        pipe = Pipeline([
            ("prep", make_preprocessor()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=hidden,
                activation="relu",
                solver="adam",
                alpha=alpha,
                learning_rate_init=lr,
                batch_size=256,
                max_iter=150,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=sd,
            )),
        ])
        t0 = time.perf_counter()
        pipe.fit(X_train, y_train)
        t_trains.append(time.perf_counter() - t0)

        preds = pipe.predict(X_val)
        f1 = f1_score(y_val, preds, average="macro")
        f1s.append(f1)
        accs.append(accuracy_score(y_val, preds))

        if f1 > best_f1:
            best_f1 = f1
            best_pipe = pipe
            best_name = f"{name} (seed={sd})"

    summary_rows.append({
        "config": name,
        "macro_f1_mean": round(float(np.mean(f1s)), 4),
        "macro_f1_std": round(float(np.std(f1s)), 4),
        "acc_mean": round(float(np.mean(accs)), 4),
        "train_time_mean_s": round(float(np.mean(t_trains)), 2),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results_mlp_progress.csv", index=False)
joblib.dump(best_pipe, "mlp_v1_kaggle.joblib")

print("Bảng tổng hợp 3 seeds (Validation):")
display(summary_df) if "display" in globals() else print(summary_df.to_string(index=False))
print(f"\nModel tốt nhất: {best_name} (Val Macro-F1 = {best_f1:.4f}) -> đã lưu mlp_v1_kaggle.joblib")


---
## Nhận xét quan trọng (đưa thẳng vào Báo cáo tiến độ 25/11!)
- Nếu mọi cấu hình MLP trên bộ **Kaggle Cyber Security Attacks** đều cho **Val Macro-F1 quanh ~0.33** (ngang mức đoán ngẫu nhiên của bài toán 3 lớp cân bằng): **code không hề sai!**
- Đây chính là **bằng chứng thực nghiệm chứng minh giả thuyết ở Mục 3 kế hoạch**: bộ dữ liệu Kaggle là dữ liệu tổng hợp (*synthetic*, Payload chứa *lorem ipsum*) trong đó nhãn `Attack Type` được sinh gần như độc lập với các đặc trưng mạng.
- **Bước tiếp theo:** Chạy y hệt pipeline `MLPClassifier` này trên bộ benchmark thật (**UNSW-NB15** hoặc **CIC-IDS2017**) của Thắng để so sánh khả năng học tín hiệu thật của 5 mô hình core!


In [ ]:
# (Tùy chọn trên Google Colab) Tải kết quả & hình ảnh về máy
try:
    from google.colab import files
    files.download("results_mlp_progress.csv")
    files.download("mlp_v1_learning_and_cm.png")
    files.download("mlp_v1_kaggle.joblib")
except ImportError:
    print("Đang chạy local — các file kết quả đã nằm ngay trong thư mục hiện tại.")
